In [3]:
# Install Hugging Face Transformers and PyTorch (if not already installed)
!pip install transformers torch

In [4]:
# Importing necessary libraries from Hugging Face Transformers and PyTorch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [5]:
# Load GPT-2 tokenizer — converts text to token IDs
print("Loading GPT-2 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Set pad token to eos token (GPT-2 has no pad token by default)
tokenizer.pad_token = tokenizer.eos_token

# Load GPT-2 language model
print("Loading GPT-2 model...")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()  # Set model to evaluation mode (disables dropout)

print("\nGPT-2 model loaded successfully!")

Loading GPT-2 tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


GPT-2 model loaded successfully!


In [6]:
def generate_response(user_input, conversation_history):
    """
    Generates a relevant response using GPT-2 with prompt engineering.

    The key idea: we format the entire conversation as a Q&A prompt.
    GPT-2 then continues the text, naturally generating an answer.

    Args:
        user_input (str): Current message from the user.
        conversation_history (str): Previous conversation turns as formatted text.

    Returns:
        response (str): The chatbot's generated reply.
        updated_history (str): Updated conversation string with new exchange.
    """

    # Build a Q&A style prompt — this guides GPT-2 to answer like an assistant
    # Including history gives the model context of previous exchanges
    prompt = (
        "The following is a conversation with an AI assistant. "
        "The assistant gives helpful, accurate, and informative answers.\n\n"
        + conversation_history
        + f"User: {user_input}\nAssistant:"
    )

    # Tokenize the prompt and convert to PyTorch tensor
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=900  # Keep input within GPT-2's 1024 token limit
    )

    input_length = inputs["input_ids"].shape[1]  # Track input length to extract only new tokens

    # Generate response tokens
    with torch.no_grad():  # Disable gradient computation for faster inference
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=100,       # Generate up to 100 new tokens for the response
            do_sample=True,           # Enable sampling for varied responses
            temperature=0.7,          # Lower = more focused/factual responses
            top_k=50,                 # Only sample from top 50 probable tokens
            top_p=0.9,                # Nucleus sampling — keeps 90% probability mass
            repetition_penalty=1.3,   # Penalize repeated words/phrases
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (skip the input prompt tokens)
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Clean up response — stop at next "User:" if model continues generating
    # This prevents the model from hallucinating further conversation turns
    if "User:" in response:
        response = response.split("User:")[0].strip()
    if "\n" in response:
        response = response.split("\n")[0].strip()

    # Fallback in case response is empty
    if not response:
        response = "I'm not sure about that. Could you rephrase or ask something else?"

    # Append this exchange to conversation history for context in next turn
    updated_history = conversation_history + f"User: {user_input}\nAssistant: {response}\n\n"

    return response, updated_history

    # Generate a response from the model
    # max_length: limits total token length to avoid overly long outputs
    # pad_token_id: suppresses warning for models without a dedicated padding token
    # do_sample: enables sampling for more varied responses
    # top_k: considers only the top 50 probable next tokens (filters unlikely words)
    # top_p: nucleus sampling — uses smallest set of tokens whose cumulative prob >= 0.95
    # temperature: controls randomness (lower = more focused, higher = more creative)
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # Decode only the newly generated tokens (exclude the input/history tokens)
    # skip_special_tokens=True removes EOS and other special markers from output
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [7]:
def run_chatbot():
    """
    Main chatbot loop — accepts user input, generates response, maintains history.
    Stops when user types 'exit' or 'quit'.
    """

    print("=" * 50)
    print("      GPT-2 AI Chatbot")
    print("  Type 'exit' or 'quit' to end the chat.")
    print("=" * 50)
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

    conversation_history = ""  # Stores full conversation as formatted string

    while True:
        user_input = input("You: ").strip()

        # Handle empty input
        if not user_input:
            print("Chatbot: Please type something!\n")
            continue

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Goodbye! Have a great day!")
            print("=" * 50)
            break

        # Generate and display response
        response, conversation_history = generate_response(user_input, conversation_history)
        print(f"\nChatbot: {response}\n")

run_chatbot()

      GPT-2 AI Chatbot
  Type 'exit' or 'quit' to end the chat.

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Chatbot: I'm the one who's here for you today! How'd ya do? You know what they say when your dog comes in to help out on his journey home from work that he can't finish yet... We're going down South tomorrow morning after lunch - well then let's get back up there together as soon we possibly could if possible so it'll be pretty safe." [applause] "Yeah?" (pause) So how did she go about doing this...? She went through all of

You: exit

Chatbot: Goodbye! Have a great day!
